# Student Performance Detector

**Goal:** predict whether a university student reports **depression** (`Depression` = Yes/No) from demographic, academic and lifestyle features (gender, age, CGPA, semester, health, living style, etc.).

> Note: despite the project name, the target here is `Depression`; academic performance (`CGPA`) is used as an input feature.

**Metrics:** the classes are imbalanced (~73% "No"), so accuracy alone is misleading. We focus on **F1 / recall for the "Yes" class** and **ROC-AUC**.

## 1. Setup

In [ ]:
# Dependencies are listed in requirements.txt:  pip install -r requirements.txt
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV,
)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    classification_report, ConfusionMatrixDisplay, RocCurveDisplay,
    roc_auc_score, f1_score, precision_recall_curve,
)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## 2. Load & inspect the data

In [ ]:
# Works whether the kernel starts in the project root or elsewhere next to data/
DATA_PATH = Path("data/raw/data.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path.cwd().parent / "data/raw/data.csv"

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

In [ ]:
print("Missing values per column:\n", df.isna().sum(), sep="")
print("\nDuplicate rows:", df.duplicated().sum())

## 3. Exploratory data analysis

In [ ]:
TARGET = "Depression"
categorical_cols = df.select_dtypes(include="object").columns.drop(TARGET).tolist()

for col in categorical_cols + [TARGET]:
    print(f"--- {col} ---")
    print(df[col].value_counts(normalize=True).round(3).to_string(), "\n")

### 3.1 Target distribution

In [ ]:
ax = sns.countplot(data=df, x=TARGET)
for p in ax.patches:
    ax.annotate(f"{p.get_height() / len(df):.0%}", (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom")
ax.set_title("Depression class balance")
plt.show()

### 3.2 Depression rate by categorical feature

In [ ]:
df["_dep"] = (df[TARGET] == "Yes").astype(int)
cat_like = categorical_cols + ["Current_semester"]

ncols = 3
nrows = int(np.ceil(len(cat_like) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.8 * nrows))
for ax, col in zip(axes.flat, cat_like):
    rate = df.groupby(col)["_dep"].mean().sort_values()
    rate.plot.barh(ax=ax, color="tab:blue")
    ax.axvline(df["_dep"].mean(), color="red", ls="--", lw=1)
    ax.set_title(col); ax.set_xlabel("Depression rate"); ax.set_ylabel("")
for ax in axes.flat[len(cat_like):]:
    ax.set_visible(False)
plt.tight_layout()
plt.show()

### 3.3 Numeric features vs. target

In [ ]:
numeric_cols = ["Age", "Current_semester", "CGPA", "Cost_without_study"]

fig, axes = plt.subplots(1, len(numeric_cols), figsize=(4.5 * len(numeric_cols), 4))
for ax, col in zip(axes, numeric_cols):
    sns.boxplot(data=df, x=TARGET, y=col, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(df[numeric_cols + ["_dep"]].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation (numeric features + target)")
plt.show()
df = df.drop(columns="_dep")

### 3.4 Key findings
- Target is imbalanced: ~73% "No", ~27% "Yes".
- **`Health_condition` almost determines the target**: "Critical" -> 100% depressed, "Good" -> 0%, "Normal" -> ~5%, "Abnormal" -> ~55%. Expect very high model scores; this suggests the data is rule-generated or that `Health_condition` partly *encodes* the outcome (possible leakage). Section 10.2 checks performance without it.
- Single students and those without a friend circle show higher depression rates; CGPA / cost / age differ only slightly between classes.

## 4. Feature definitions

In [ ]:
numeric_features = ["Age", "Current_semester", "CGPA", "Cost_without_study"]

# Categories with a natural order -> ordinal encoding (low -> high)
ordinal_features = ["Family_background", "Health_condition"]
ordinal_categories = [
    ["Lower Middle class", "Middle class", "Rich class"],
    ["Critical", "Abnormal", "Normal", "Good"],
]

# Categories without order -> one-hot encoding
nominal_features = ["Gender", "Relationship_status", "Marital_status",
                    "Living_style", "friend_circle", "Home_division"]

X = df[numeric_features + ordinal_features + nominal_features]
y = (df[TARGET] == "Yes").astype(int)   # 1 = depressed
print(X.shape, y.shape, "positive rate:", round(y.mean(), 3))

## 5. Train / test split (stratified)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Positive rate train/test:", round(y_train.mean(), 3), round(y_test.mean(), 3))

## 6. Preprocessing pipeline
Encoding and scaling live inside a `Pipeline`, so they are fit only on training folds (no data leakage).

In [ ]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("ord", OrdinalEncoder(categories=ordinal_categories), ordinal_features),
    ("nom", OneHotEncoder(handle_unknown="ignore", drop="if_binary"), nominal_features),
])

def make_pipeline(model):
    return Pipeline([("prep", preprocessor), ("model", model)])

## 7. Baseline & model comparison (5-fold stratified CV)

In [ ]:
models = {
    "Dummy (most frequent)": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(class_weight="balanced", max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                            random_state=RANDOM_STATE, n_jobs=-1),
    "Hist Gradient Boosting": HistGradientBoostingClassifier(class_weight="balanced",
                                                             random_state=RANDOM_STATE),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

rows = []
for name, model in models.items():
    scores = cross_validate(make_pipeline(model), X_train, y_train, cv=cv, scoring=scoring)
    rows.append({"model": name, **{m: scores[f"test_{m}"].mean() for m in scoring}})

results = pd.DataFrame(rows).set_index("model").sort_values("f1", ascending=False)
results.round(3)

## 8. Hyperparameter tuning (best tree model)

In [ ]:
param_dist = {
    "model__n_estimators": [200, 300, 500],
    "model__max_depth": [None, 5, 10, 20],
    "model__min_samples_leaf": [1, 2, 5, 10],
    "model__max_features": ["sqrt", "log2", 0.5],
}

search = RandomizedSearchCV(
    make_pipeline(RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)),
    param_distributions=param_dist, n_iter=20, scoring="f1", cv=cv,
    random_state=RANDOM_STATE, n_jobs=-1,
)
search.fit(X_train, y_train)
print("Best CV F1:", round(search.best_score_, 3))
print("Best params:", search.best_params_)

## 9. Final evaluation on the test set

In [ ]:
# Pick the tuned model if it beats the best untuned CV F1, else refit the best untuned one
best_untuned = results.drop("Dummy (most frequent)").index[0]
if search.best_score_ >= results.loc[best_untuned, "f1"]:
    final_name, final_model = "Random Forest (tuned)", search.best_estimator_
else:
    final_name, final_model = best_untuned, make_pipeline(models[best_untuned]).fit(X_train, y_train)

y_pred = final_model.predict(X_test)
y_proba = final_model.predict_proba(X_test)[:, 1]

print("Final model:", final_name)
print(classification_report(y_test, y_pred, target_names=["No", "Yes"]))
print("ROC-AUC:", round(roc_auc_score(y_test, y_proba), 3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=["No", "Yes"], ax=axes[0], colorbar=False)
axes[0].set_title("Confusion matrix")
RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[1])
axes[1].plot([0, 1], [0, 1], "k--", lw=1)
axes[1].set_title("ROC curve")
plt.tight_layout()
plt.show()

### 9.1 Decision threshold
The default threshold is 0.5. Since missing a depressed student is costly, inspect how F1 changes with the threshold.
_(For an unbiased choice, tune the threshold on a validation split / CV predictions rather than the test set.)_

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, y_proba)
f1s = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
best_t = thresholds[np.argmax(f1s)]

plt.plot(thresholds, precision[:-1], label="precision")
plt.plot(thresholds, recall[:-1], label="recall")
plt.plot(thresholds, f1s, label="F1")
plt.axvline(best_t, color="gray", ls="--")
plt.xlabel("threshold"); plt.legend(); plt.title("Precision / recall vs. threshold")
plt.show()
print(f"Best F1 threshold: {best_t:.2f}  (F1 = {f1s.max():.3f} vs {f1_score(y_test, y_pred):.3f} at 0.5)")

## 10. Interpretation — which features matter?

In [ ]:
perm = permutation_importance(final_model, X_test, y_test, scoring="f1",
                              n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
importances = pd.Series(perm.importances_mean, index=X_test.columns).sort_values()

importances.plot.barh(figsize=(7, 5), title="Permutation importance (drop in F1)")
plt.show()

In [ ]:
# Logistic regression coefficients (direction of effect per encoded feature)
lr = make_pipeline(LogisticRegression(class_weight="balanced", max_iter=1000)).fit(X_train, y_train)
coefs = pd.Series(lr.named_steps["model"].coef_[0],
                  index=lr.named_steps["prep"].get_feature_names_out()).sort_values()

coefs.plot.barh(figsize=(7, 8), color=np.where(coefs > 0, "tab:red", "tab:blue"),
                title="Logistic regression coefficients (+ = more likely depressed)")
plt.show()

### 10.1 SHAP values
SHAP explains each prediction as a sum of per-feature contributions (in log-odds), giving both a global ranking and per-student explanations.

In [ ]:
import shap

prep = final_model.named_steps["prep"]
clf = final_model.named_steps["model"]

X_test_enc = prep.transform(X_test)
if hasattr(X_test_enc, "toarray"):          # ColumnTransformer may return a sparse matrix
    X_test_enc = X_test_enc.toarray()
X_test_enc = pd.DataFrame(X_test_enc, columns=prep.get_feature_names_out(), index=X_test.index)

explainer = shap.TreeExplainer(clf)
shap_values = explainer(X_test_enc)
if shap_values.values.ndim == 3:            # RandomForest returns one set per class -> keep "Yes"
    shap_values = shap_values[:, :, 1]

In [ ]:
# Global: overall importance and direction of effect
shap.plots.bar(shap_values, max_display=15)
shap.plots.beeswarm(shap_values, max_display=15)

In [ ]:
# How a numeric feature's value shifts the prediction
shap.plots.scatter(shap_values[:, "num__CGPA"], color=shap_values)

In [ ]:
# Local: why the model flagged one particular depressed student
idx = int(np.flatnonzero(y_test.to_numpy() == 1)[0])
shap.plots.waterfall(shap_values[idx], max_display=12)

### 10.2 Ablation: how well can we do without `Health_condition`?
If health status is recorded *because of* (or together with) the depression diagnosis, it shouldn't be used as a predictor.

In [ ]:
pre_no_health = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("ord", OrdinalEncoder(categories=[ordinal_categories[0]]), ["Family_background"]),
    ("nom", OneHotEncoder(handle_unknown="ignore", drop="if_binary"), nominal_features),
])
no_health = Pipeline([("prep", pre_no_health),
                      ("model", HistGradientBoostingClassifier(class_weight="balanced", random_state=RANDOM_STATE))])
X_train_nh = X_train.drop(columns="Health_condition")
scores = cross_validate(no_health, X_train_nh, y_train, cv=cv, scoring=["f1", "roc_auc"])
print(f"Without Health_condition -> CV F1: {scores['test_f1'].mean():.3f}, ROC-AUC: {scores['test_roc_auc'].mean():.3f}")

## 11. Conclusion
- **Best model:** _…_ (test F1 = _…_, ROC-AUC = _…_) vs. baseline F1 = 0.
- **Most influential features:** _…_
- **Limitations:** survey-style data, class imbalance, possible self-report bias; association ≠ causation.
- **Next steps:** tune threshold on CV predictions, try feature interactions, calibrate probabilities, save the pipeline (`joblib.dump(final_model, "models/model.joblib")`).